# B2B Customer Churn — Exploratory Data Analysis

**Purpose:** Understand business patterns, data quality, and churn drivers before feature engineering and modeling (XGBoost + SHAP).

**Dataset:** `../data/customer_churn_business_dataset.csv`

**Scope:** Descriptive statistics and visualizations only — no model training in this notebook.


In [ ]:
# --- Imports & display configuration ---
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)

sns.set_theme(style="whitegrid", context="notebook", palette="deep")
plt.rcParams.update({
    "figure.figsize": (10, 6),
    "figure.dpi": 100,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
})

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# Constants
DATA_PATH = Path("../data/customer_churn_business_dataset.csv")
TARGET = "churn"
ID_COLS = ["customer_id"]

# Features explicitly reviewed in business deep-dive section
KEY_FEATURES = [
    "tenure_months",
    "monthly_logins",
    "avg_session_time",
    "usage_growth_rate",
    "support_tickets",
    "csat_score",
    "nps_score",
    "payment_failures",
    "contract_type",
    "customer_segment",
]


In [ ]:
# Load dataset
df = pd.read_csv(DATA_PATH)
df.head()


## 1. Dataset overview

We inspect structure, types, summary statistics, and data-quality issues (missing values, duplicates) that would block reliable modeling.


In [ ]:
print("Shape (rows, columns):", df.shape)
print("\nColumn names:")
print(df.columns.tolist())


In [ ]:
print("Data types:\n")
display(df.dtypes.to_frame("dtype"))


In [ ]:
df.describe(include="all").T


In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
quality = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
quality[quality["missing_count"] > 0]
if quality["missing_count"].sum() == 0:
    print("No missing values detected.")
else:
    display(quality[quality["missing_count"] > 0])


In [ ]:
duplicate_rows = df.duplicated().sum()
duplicate_ids = df.duplicated(subset=ID_COLS).sum()
print(f"Fully duplicate rows: {duplicate_rows}")
print(f"Duplicate {ID_COLS[0]} values: {duplicate_ids}")


### Business insight — data quality

- **Record volume** sets how stable segment-level churn rates will be; thin slices need smoothing or grouping at modeling time.
- **Complete cases** (no missing values) simplify preprocessing but do not guarantee signal quality — validate plausibility of extremes.
- **Unique customer IDs** are required for B2B account-level churn; duplicate keys would leak information or double-count revenue at risk.
- Next we profile the **target (churn)** and separate **numeric vs categorical** predictors for tailored plots.


## 2. Target variable — churn

Churn prevalence drives evaluation metrics (precision/recall), class weights, and intervention capacity planning.


In [ ]:
churn_counts = df[TARGET].value_counts().sort_index()
churn_pct = df[TARGET].mean() * 100

print("Churn distribution:")
print(churn_counts.rename({0: "Retained (0)", 1: "Churned (1)"}))
print(f"\nChurn rate: {churn_pct:.2f}%")
print(f"Class ratio (retained : churned): {churn_counts[0] / churn_counts[1]:.1f} : 1")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=df, x=TARGET, ax=axes[0])
axes[0].set_title("Churn class counts")
axes[0].set_xticklabels(["Retained", "Churned"])

labels = ["Retained", "Churned"]
sizes = churn_counts.values
axes[1].pie(sizes, labels=labels, autopct="%1.1f%%", startangle=90, colors=sns.color_palette()[:2])
axes[1].set_title("Churn class share")

plt.tight_layout()
plt.show()


### Business insight — class imbalance

- A **minority churn class** is typical in B2B SaaS; accuracy alone is misleading — prioritize recall on churners or expected value of saves.
- Operations teams should align outreach capacity with the **absolute number of churned accounts** in the holdout period, not only the rate.
- Plan for **stratified splits** and possibly `scale_pos_weight` / resampling when modeling; SHAP will still rank drivers on the full feature set.


## 3. Numerical vs categorical features

Separating feature types guides encoding (one-hot / target encoding) and choice of plots (distributions vs segment bars).


In [ ]:
NUM_COLS = df.select_dtypes(include=[np.number]).columns.drop(TARGET, errors="ignore").tolist()
CAT_COLS = [c for c in df.columns if c not in NUM_COLS + ID_COLS + [TARGET]]

print("Numerical features:", NUM_COLS)
print("Categorical features:", CAT_COLS)


### Business insight — feature typing

- **Engagement and satisfaction metrics** (logins, session time, CSAT, NPS) are continuous levers CS teams can influence.
- **Contract type and segment** define go-to-market motion and renewal playbook — expect different churn baselines by category.
- Identifier columns must be **excluded from training** but retained for joining to CRM exports.


## 4. Univariate distributions

Histograms and boxplots reveal skew, heavy tails, and outliers that affect scaling, binning, and tree depth during modeling.


In [ ]:
n_num = len(NUM_COLS)
ncols = 3
nrows = int(np.ceil(n_num / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows))
axes = axes.flatten()

for ax, col in zip(axes, NUM_COLS):
    sns.histplot(df[col], kde=True, ax=ax, bins=30)
    ax.set_title(f"Histogram — {col}")

for ax in axes[n_num:]:
    ax.set_visible(False)

plt.suptitle("Numerical feature distributions", y=1.01, fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows))
axes = axes.flatten()

for ax, col in zip(axes, NUM_COLS):
    sns.boxplot(y=df[col], ax=ax, color=sns.color_palette()[0])
    ax.set_title(f"Boxplot — {col}")

for ax in axes[n_num:]:
    ax.set_visible(False)

plt.suptitle("Numerical feature boxplots (outlier check)", y=1.01, fontsize=14)
plt.tight_layout()
plt.show()


### Business insight — univariate shapes

- **Right-skewed tenure and ticket counts** are common: most accounts are newer or low-touch, with a long tail of veterans or escalations.
- Boxplot whisker exceedances flag **outliers** — validate whether they are data errors, enterprise whales, or distressed accounts before dropping.
- Strong skew suggests **log transforms or quantile binning** may help linear models; tree models (XGBoost) are more robust but still benefit from sane ranges.


## 5. Correlation structure

Pairwise correlations among numeric features highlight redundancy (multicollinearity) and linear association with churn.


In [ ]:
corr = df[NUM_COLS + [TARGET]].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    square=True,
    linewidths=0.5,
)
plt.title("Correlation heatmap (numeric features + churn)")
plt.tight_layout()
plt.show()


In [ ]:
# Features most correlated with churn (absolute value)
churn_corr = corr[TARGET].drop(TARGET).abs().sort_values(ascending=False)
print("Correlation with churn (|r|):")
print(churn_corr)

# Highly correlated feature pairs (|r| > 0.6), excluding churn
pairs = []
for i, a in enumerate(NUM_COLS):
    for b in NUM_COLS[i + 1:]:
        r = corr.loc[a, b]
        if abs(r) > 0.6:
            pairs.append((a, b, r))

if pairs:
    print("\nFeature pairs with |r| > 0.6:")
    for a, b, r in sorted(pairs, key=lambda x: -abs(x[2])):
        print(f"  {a} <-> {b}: {r:.3f}")
else:
    print("\nNo numeric pairs with |r| > 0.6 (excluding churn).")


### Business insight — correlations

- Features with the **largest |r| vs churn** are hypotheses for drivers (e.g., satisfaction, usage trend) — confirm with segment plots before causal claims.
- **High inter-feature correlation** inflates coefficient variance in linear models; trees handle overlap better but SHAP values may split credit across twins.
- Correlation is **not causation**; contract terms and segment often confound engagement metrics in B2B settings.


## 6. Churn vs numerical features

Compare retained vs churned distributions on all numeric predictors.


In [ ]:
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows))
axes = axes.flatten()

for ax, col in zip(axes, NUM_COLS):
    sns.boxplot(data=df, x=TARGET, y=col, ax=ax)
    ax.set_xticklabels(["Retained", "Churned"])
    ax.set_title(col)

for ax in axes[n_num:]:
    ax.set_visible(False)

plt.suptitle("Churn vs numerical features (boxplots)", y=1.01, fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# Mean difference (churned minus retained) for quick effect direction
mean_diff = df.groupby(TARGET)[NUM_COLS].mean().T
mean_diff.columns = ["retained_mean", "churned_mean"]
mean_diff["delta_churned_minus_retained"] = mean_diff["churned_mean"] - mean_diff["retained_mean"]
mean_diff.sort_values("delta_churned_minus_retained", key=abs, ascending=False)


### Business insight — numeric churn patterns

- **Lower tenure and CSAT** among churners often indicate failed onboarding or unresolved value gaps — early-life saves have highest ROI.
- **Negative usage growth** among churners signals product disengagement before cancellation; product-led growth teams should trigger plays on usage drops.
- **More support tickets** can mean either high investment (saveable) or frustration (toxic); combine with CSAT/NPS when prioritizing outreach.


## 7. Churn vs categorical features

Churn rate by contract and segment informs packaging, success coverage, and renewal strategy.


In [ ]:
fig, axes = plt.subplots(1, len(CAT_COLS), figsize=(6 * len(CAT_COLS), 5))
if len(CAT_COLS) == 1:
    axes = [axes]

for ax, col in zip(axes, CAT_COLS):
    rate = df.groupby(col)[TARGET].mean().sort_values(ascending=False)
    sns.barplot(x=rate.index, y=rate.values, ax=ax, hue=rate.index, legend=False)
    ax.set_title(f"Churn rate by {col}")
    ax.set_ylabel("Churn rate")
    ax.set_ylim(0, rate.max() * 1.25)
    for i, v in enumerate(rate.values):
        ax.text(i, v + 0.002, f"{v:.1%}", ha="center", fontsize=10)

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, len(CAT_COLS), figsize=(6 * len(CAT_COLS), 5))
if len(CAT_COLS) == 1:
    axes = [axes]

for ax, col in zip(axes, CAT_COLS):
    ct = pd.crosstab(df[col], df[TARGET], normalize="index")
    ct.plot(kind="bar", stacked=True, ax=ax, color=sns.color_palette()[:2])
    ax.set_title(f"Churn mix by {col}")
    ax.set_ylabel("Proportion")
    ax.legend(["Retained", "Churned"], title=TARGET)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right")

plt.tight_layout()
plt.show()


### Business insight — categorical churn patterns

- **Monthly contracts** usually churn more than multi-year deals — flexibility trades off commitment; upsell to annual can reduce logo churn.
- **SMB segments** often show higher churn than enterprise; success models should be tiered (tech-touch vs high-touch).
- Use these rates for **benchmarking** dashboards; model scores will personalize risk within each segment.


## 8. Key feature deep-dive

Focused analysis on the ten business-critical fields most tied to retention levers and commercial terms.


In [ ]:
key_num = [c for c in KEY_FEATURES if c in NUM_COLS]
key_cat = [c for c in KEY_FEATURES if c in CAT_COLS]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for ax, col in zip(axes, key_num):
    sns.kdeplot(data=df, x=col, hue=TARGET, hue_order=[0, 1], fill=True, alpha=0.4, ax=ax)
    ax.set_title(col)
    ax.legend(["Retained", "Churned"], title="Churn")

for ax in axes[len(key_num):]:
    ax.set_visible(False)

plt.suptitle("Key numerical features — density by churn", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Churn rate & summary stats for key numerics
key_summary = (
    df.groupby(TARGET)[key_num]
    .agg(["mean", "median", "std"])
    .round(3)
)
display(key_summary)


In [ ]:
for col in key_cat:
    display(
        df.groupby(col)[TARGET]
        .agg(churn_rate="mean", accounts="count")
        .sort_values("churn_rate", ascending=False)
        .assign(churn_rate_pct=lambda x: (x["churn_rate"] * 100).round(2))
    )


In [ ]:
# Pairwise business views: satisfaction vs engagement
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(
    data=df.sample(min(2000, len(df)), random_state=42),
    x="csat_score",
    y="usage_growth_rate",
    hue=TARGET,
    alpha=0.5,
    ax=axes[0],
)
axes[0].set_title("CSAT vs usage growth (sample)")

sns.scatterplot(
    data=df.sample(min(2000, len(df)), random_state=42),
    x="tenure_months",
    y="monthly_logins",
    hue=TARGET,
    alpha=0.5,
    ax=axes[1],
)
axes[1].set_title("Tenure vs monthly logins (sample)")

plt.tight_layout()
plt.show()


### Business insight — key drivers (hypotheses)

| Signal | Typical churn pattern | Commercial implication |
|--------|----------------------|-------------------------|
| `tenure_months` | Shorter tenure → higher churn | Strengthen first 90-day onboarding and value proof |
| `usage_growth_rate` | Flat or negative growth → risk | Trigger success plays when usage MoM declines |
| `csat_score` / `nps_score` | Lower scores → risk | Close the loop on detractors before renewal |
| `support_tickets` | Elevated volume → risk (if CSAT low) | Escalation QA and root-cause fixes on repeat issues |
| `payment_failures` | More failures → risk | Billing outreach and payment method updates |
| `contract_type` | Monthly highest rate | Incentivize longer commitments at signup/renewal |
| `customer_segment` | SMB often highest rate | Segment-specific playbooks and coverage models |

Engagement metrics (`monthly_logins`, `avg_session_time`) refine **when** to act; satisfaction and contract terms help explain **why** accounts leave.


## 9. Synthesis — drivers, outliers, skew, redundancy

Consolidated diagnostics to feed feature engineering and modeling choices (no training in this notebook).


In [ ]:
# Skewness (|skew| > 1 suggests strong asymmetry)
skewness = df[NUM_COLS].skew().sort_values(key=abs, ascending=False)
skew_report = pd.DataFrame({
    "skewness": skewness,
    "abs_skew": skewness.abs(),
    "highly_skewed": skewness.abs() > 1,
})
skew_report


In [ ]:
# Outliers via IQR rule per numeric feature
outlier_rows = pd.DataFrame(index=NUM_COLS, columns=["outlier_count", "outlier_pct"])

for col in NUM_COLS:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    mask = (df[col] < lower) | (df[col] > upper)
    outlier_rows.loc[col, "outlier_count"] = int(mask.sum())
    outlier_rows.loc[col, "outlier_pct"] = round(mask.mean() * 100, 2)

outlier_rows = outlier_rows.astype({"outlier_count": int, "outlier_pct": float})
outlier_rows.sort_values("outlier_pct", ascending=False)


In [ ]:
# Rank potential churn drivers: correlation + standardized mean shift
z_shift = (
    df.groupby(TARGET)[NUM_COLS].mean().diff().iloc[-1]
    / df[NUM_COLS].std()
).abs().sort_values(ascending=False)

driver_table = pd.DataFrame({
    "abs_corr_with_churn": churn_corr,
    "abs_std_mean_shift": z_shift,
})
driver_table["avg_rank_signal"] = (
    driver_table.rank(ascending=False).mean(axis=1)
)
driver_table.sort_values("avg_rank_signal")


### Business insight — synthesis & next steps

**Likely churn drivers (from EDA, to validate in modeling):**
- Satisfaction and perceived value (`csat_score`, `nps_score`)
- Product adoption trajectory (`usage_growth_rate`, `monthly_logins`, `avg_session_time`)
- Relationship depth (`tenure_months`) and friction (`support_tickets`, `payment_failures`)
- Commercial structure (`contract_type`, `customer_segment`)

**Data preparation recommendations (before XGBoost):**
1. Encode categoricals (`contract_type`, `customer_segment`); keep definitions stable for production API.
2. Consider **winsorizing or flagging outliers** on ticket counts and session time if SHAP shows domination by extremes.
3. Address **class imbalance** in training configuration; monitor precision@k for outreach lists.
4. Engineer **interaction features** (e.g., tickets × low CSAT) where business logic supports non-linear risk.
5. Hold out a **time-based or stratified test set** if future data includes cohort drift.

**Explicitly out of scope here:** model fitting, hyperparameter tuning, SHAP on trained estimators, and deployment — covered in later pipeline notebooks.
